In [ ]:
from datasets import load_dataset

In [ ]:
HF_TOKEN = "" # Add your hf token here

# Q1

In [ ]:
ds = load_dataset("rojagtap/bookcorpus")

In [ ]:
total_samples = len(ds['train'])
indices = list(range(0, total_samples, 7))
ds_copy = ds['train'].select(indices)

In [ ]:
!pip install tokenizers

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.normalizers import Lowercase
from tokenizers import decoders
from tokenizers.trainers import BpeTrainer # Import BpeTrainer

# Define special tokens
special_tokens = {
    "unk_token": "[UNK]",
    "bos_token": "[GO]",
    "pad_token": "[PAD]",
    "eos_token": "[EOS]"
}

# Initialize a BPE tokenizer
tokenizer = Tokenizer(BPE(unk_token=special_tokens["unk_token"]))

# Set the normalizer to Lowercase
tokenizer.normalizer = Lowercase()

# Set the pre-tokenizer to Whitespace
tokenizer.pre_tokenizer = Whitespace()

# Set the decoder
tokenizer.decoder = decoders.BPEDecoder() # Changed to BPEDecoder

# Prepare the dataset for training
def batch_iterator(dataset, batch_size=1000):
    for i in range(0, len(dataset), batch_size):
        yield dataset[i : i + batch_size]["text"]

# Train the tokenizer
vocab_size = 5000 # You can vary this vocabulary size as requested

# Initialize BpeTrainer
trainer = BpeTrainer(
    vocab_size=vocab_size,
    # min_frequency=2,
    special_tokens=list(special_tokens.values())
)

tokenizer.train_from_iterator(
    batch_iterator(ds_copy),
    trainer=trainer,
    length=len(ds_copy) # Added length for better progress reporting
)

print(f"Tokenizer trained with vocab size: {vocab_size}")

Tokenizer trained with vocab size: 5000


In [ ]:
# Removed import tokenizers.processors as TemplateProcessing is no longer used

# PostProcessing: None as per request, so removing post_processor setup
# Removed go_id, eos_id checks and TemplateProcessing setup

# Input text to tokenize
input_text = "SEBI study finds 93% of individual F&O traders made losses between FY22 and FY24."

# Tokenize the input text
output = tokenizer.encode(input_text)

print(f"Original text: {input_text}")
print(f"Tokens: {output.tokens}")
print(f"IDs: {output.ids}")
print(f"Decoded text: {tokenizer.decode(output.ids)}")

Original text: SEBI study finds 93% of individual F&O traders made losses between FY22 and FY24.
Tokens: ['se', 'bi', 'study', 'find', 's', '9', '3', '%', 'of', 'indi', 'vid', 'ual', 'f', '&', 'o', 'trad', 'ers', 'made', 'lo', 'sses', 'between', 'fy', '2', '2', 'and', 'fy', '2', '4', '.']
IDs: [108, 1382, 3444, 564, 65, 33, 27, 13, 100, 4765, 1583, 1069, 52, 14, 61, 4996, 245, 440, 129, 3690, 716, 4679, 26, 26, 91, 4679, 26, 28, 22]
Decoded text: sebistudyfinds93%ofindividualf&otradersmadelossesbetweenfy22andfy24.


### Retraining Tokenizer with Varying Vocabulary Sizes

In [ ]:
# List of vocabulary sizes to test
vocab_sizes_to_test = [10000, 15000, 32000]

for current_vocab_size in vocab_sizes_to_test:
    print(f"\n--- Training tokenizer with vocab_size: {current_vocab_size} ---")

    # Re-initialize BpeTrainer with the new vocab_size
    trainer = BpeTrainer(
        vocab_size=current_vocab_size,
        min_frequency=2,
        special_tokens=list(special_tokens.values())
    )

    # Re-initialize the tokenizer to ensure a fresh start for each training run
    tokenizer = Tokenizer(BPE(unk_token=special_tokens["unk_token"]))
    tokenizer.normalizer = Lowercase()
    tokenizer.pre_tokenizer = Whitespace()
    tokenizer.decoder = decoders.BPEDecoder()

    tokenizer.train_from_iterator(
        batch_iterator(ds_copy),
        trainer=trainer,
        length=len(ds_copy)
    )

    print(f"Tokenizer trained with vocab size: {current_vocab_size}")

    # Tokenize the input text with the newly trained tokenizer
    output = tokenizer.encode(input_text)

    print(f"Original text: {input_text}")
    print(f"Tokens: {output.tokens}")
    print(f"IDs: {output.ids}")
    print(f"Decoded text: {tokenizer.decode(output.ids)}")



--- Training tokenizer with vocab_size: 10000 ---
Tokenizer trained with vocab size: 10000
Original text: SEBI study finds 93% of individual F&O traders made losses between FY22 and FY24.
Tokens: ['se', 'bi', 'study', 'finds', '9', '3', '%', 'of', 'individual', 'f', '&', 'o', 'trad', 'ers', 'made', 'lo', 'sses', 'between', 'fy', '22', 'and', 'fy', '24', '.']
IDs: [108, 1382, 3444, 6294, 33, 27, 13, 100, 5652, 52, 14, 61, 4996, 245, 440, 129, 3690, 716, 4679, 8584, 91, 4679, 8447, 22]
Decoded text: sebistudyfinds93%ofindividualf&otradersmadelossesbetweenfy22andfy24.

--- Training tokenizer with vocab_size: 15000 ---
Tokenizer trained with vocab size: 15000
Original text: SEBI study finds 93% of individual F&O traders made losses between FY22 and FY24.
Tokens: ['se', 'bi', 'study', 'finds', '9', '3', '%', 'of', 'individual', 'f', '&', 'o', 'trad', 'ers', 'made', 'lo', 'sses', 'between', 'fy', '22', 'and', 'fy', '24', '.']
IDs: [108, 1382, 3444, 6294, 33, 27, 13, 100, 5652, 52, 14, 61, 4

In [ ]:
print(len([108, 1382, 3444, 6294, 33, 27, 13, 100, 5652, 52, 14, 61, 4996, 245, 440, 129, 3690, 716, 4679, 8584, 91, 4679, 8447, 22]))
print(len([108, 1382, 3444, 6294, 33, 27, 13, 100, 5652, 52, 14, 61, 4996, 245, 440, 129, 3690, 716, 4679, 8584, 91, 4679, 8447, 22]))
print(len([108, 1382, 3444, 6294, 33, 27, 13, 100, 5652, 52, 14, 61, 20987, 440, 16079, 716, 4679, 8584, 91, 4679, 8447, 22]))

24
24
22


In [ ]:
# Load the 'imdb' dataset, skipping the 'unsupervised' split
imdb_dataset = load_dataset("stanfordnlp/imdb", split=['train', 'test'])

In [ ]:
def tokenize_function(examples):
    encoding = tokenizer.encode(examples["text"])
    return {"tokens": encoding.tokens, "ids": encoding.ids}

# Tokenize the 'train' split
tokenized_imdb_train = imdb_dataset[0].map(tokenize_function, batched=False) # imdb_dataset[0] is the 'train' split
# Tokenize the 'test' split
tokenized_imdb_test = imdb_dataset[1].map(tokenize_function, batched=False) # imdb_dataset[1] is the 'test' split

print("Tokenization of IMDB dataset complete.")

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Tokenization of IMDB dataset complete.


In [ ]:
# Display a sample of the tokenized 'train' split
print("\nSample of tokenized IMDB train dataset:")
for i in range(3):
    print(f"Original Text: {imdb_dataset[0][i]['text'][:100]}...") # Display first 100 chars
    print(f"Tokens: {tokenized_imdb_train[i]['tokens'][:20]}...") # Display first 20 tokens
    print(f"IDs: {tokenized_imdb_train[i]['ids'][:20]}...") # Display first 20 IDs
    print("---")


Sample of tokenized IMDB train dataset:
Original Text: I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it w...
Tokens: ['i', 'ren', 'ted', 'i', 'am', 'curious', '-', 'yellow', 'from', 'my', 'video', 'store', 'because', 'of', 'all', 'the', 'cont', 'ro', 'ver', 'sy']...
IDs: [55, 532, 218, 55, 186, 3522, 21, 2831, 200, 132, 4740, 2359, 449, 100, 133, 79, 509, 115, 134, 985]...
---
Original Text: "I Am Curious: Yellow" is a risible and pretentious steaming pile. It doesn't matter what one's poli...
Tokens: ['[UNK]', 'i', 'am', 'curious', ':', 'yellow', '[UNK]', 'is', 'a', 'ris', 'ible', 'and', 'pre', 'tenti', 'ous', 'ste', 'aming', 'pile', '.', 'it']...
IDs: [0, 55, 186, 3522, 34, 2831, 0, 92, 47, 931, 2102, 91, 310, 4501, 294, 438, 2701, 3841, 22, 96]...
---
Original Text: If only to avoid making this type of film in the future. This film is interesting as an experiment b...
Tokens: ['if', 'only', 'to', 'avoid', 'making', 'this', 'type', '

In [ ]:
total_tokens_train = sum(len(entry['ids']) for entry in tokenized_imdb_train)
total_tokens_test = sum(len(entry['ids']) for entry in tokenized_imdb_test)
total_tokens_imdb = total_tokens_train + total_tokens_test

print(f"Total tokens in IMDB train dataset: {total_tokens_train}")
print(f"Total tokens in IMDB test dataset: {total_tokens_test}")
print(f"Total tokens in entire IMDB dataset: {total_tokens_imdb}")

Total tokens in IMDB train dataset: 9109520
Total tokens in IMDB test dataset: 8896934
Total tokens in entire IMDB dataset: 18006454


## Notebook Summary: Deep Learning Practice with Tokenizers

This notebook demonstrates the process of building and evaluating a Byte Pair Encoding (BPE) tokenizer using the `tokenizers` library, focusing on practical implementation with both the BookCorpus and IMDB datasets.

### Theory and Mathematical Concepts

#### 1. Dataset Preparation and Sampling
*   **BookCorpus Dataset:** A large dataset used for language modeling. For efficiency, only every 7th sample is selected to create a smaller, representative subset for tokenizer training (approximately 10.5 million samples from the original 74 million). This is a form of data subsampling to manage computational resources while retaining data diversity.
*   **IMDB Dataset:** A dataset commonly used for sentiment analysis, consisting of movie reviews. It is explicitly loaded with 'train' and 'test' splits, excluding the 'unsupervised' portion to focus on supervised learning tasks if applicable, or simply reduce data volume.

#### 2. Byte Pair Encoding (BPE) Tokenization
*   **Principle:** BPE is a subword tokenization algorithm. It works by iteratively merging the most frequent adjacent character pairs into new tokens until a desired vocabulary size is reached. This allows the tokenizer to handle out-of-vocabulary (OOV) words by breaking them down into known subword units.
*   **Normalizer (Lowercase):** Before tokenization, all text is converted to lowercase. This helps reduce vocabulary size and treats words like "The" and "the" as the same token, simplifying the model's learning process.
*   **Pre-tokenizer (Whitespace):** The text is initially split into words based on whitespace. BPE then operates on these words or character sequences. This is crucial for defining initial token boundaries.
*   **Special Tokens:** Specific tokens are added to the vocabulary to denote special semantic meanings or boundaries, such as:
    *   `[GO]` (Go/Beginning of Sentence)
    *   `[UNK]` (Unknown): For tokens not present in the vocabulary.
    *   `[PAD]` (Padding): To standardize sequence lengths.
    *   `[EOS]` (End of Sentence)
*   **Decoder (BPEDecoder):** Responsible for reconstructing the original text from token IDs. With a `Whitespace` pre-tokenizer and `BPEDecoder` (without explicit post-processing), the decoded text will appear concatenated without spaces. This is because the tokenizer handles spaces as part of the token boundaries during pre-tokenization but doesn't re-insert them during simple decoding unless specific post-processing rules are applied.
*   **Vocabulary Size:** The number of unique tokens the tokenizer can produce. Varying vocabulary sizes (e.g., 10K, 15K, 32K) directly impacts the granularity of tokenization:
    *   **Smaller vocab:** More subword tokens, shorter tokens, more common words are split (e.g., 'traders' -> 'trad', 'ers').
    *   **Larger vocab:** Fewer, longer tokens, more common words are represented as single tokens (e.g., 'traders' as a single token). This often leads to more efficient representation for frequent words.

### Code Implementation and Practice

#### 1. Environment Setup
*   Installation of `tokenizers` library using `!pip install tokenizers`.
*   Setting a Hugging Face token (`HF_TOKEN`) for dataset access.

#### 2. Dataset Loading
*   `load_dataset("rojagtap/bookcorpus")`: Loads the BookCorpus dataset.
*   `ds['train'].select(indices)`: Selects every 7th sample from the BookCorpus training split.
*   `load_dataset("stanfordnlp/imdb", split=['train', 'test'])`: Loads the IMDB dataset, specifically the train and test splits.

#### 3. Tokenizer Initialization and Configuration
*   `from tokenizers import Tokenizer`: Imports the main `Tokenizer` class.
*   `from tokenizers.models import BPE`: Imports the BPE model.
*   `from tokenizers.pre_tokenizers import Whitespace`: Imports the whitespace pre-tokenizer.
*   `from tokenizers.normalizers import Lowercase`: Imports the lowercase normalizer.
*   `from tokenizers import decoders`: Imports decoders.
*   `tokenizer = Tokenizer(BPE(unk_token=special_tokens["unk_token"]))`: Initializes the tokenizer with BPE model and an unknown token.
*   `tokenizer.normalizer = Lowercase()`: Sets the normalizer.
*   `tokenizer.pre_tokenizer = Whitespace()`: Sets the pre-tokenizer.
*   `tokenizer.decoder = decoders.BPEDecoder()`: Sets the decoder to BPEDecoder.

#### 4. Tokenizer Training
*   `batch_iterator` function: A Python generator that yields batches of text from the dataset. This is essential for training the tokenizer from a large dataset efficiently.
*   `from tokenizers.trainers import BpeTrainer`: Imports the BPE trainer class.
*   `trainer = BpeTrainer(...)`: Initializes the trainer with `vocab_size`, `min_frequency`, and `special_tokens`.
*   `tokenizer.train_from_iterator(batch_iterator(ds_copy), trainer=trainer, length=len(ds_copy))`: Trains the BPE tokenizer using the prepared BookCorpus subset.

#### 5. Tokenization and Decoding Sample Text
*   `input_text = "SEBI study finds..."`: Defines a sample text.
*   `output = tokenizer.encode(input_text)`: Tokenizes the input text, returning an `Encoding` object containing tokens and their IDs.
*   `tokenizer.decode(output.ids)`: Decodes the token IDs back to text.

#### 6. Retraining with Varying Vocabulary Sizes
*   A loop iterates through `vocab_sizes_to_test = [10000, 15000, 32000]`.
*   Inside the loop, the tokenizer and trainer are re-initialized and retrained with each specified vocabulary size.
*   The same sample text is tokenized and decoded for each vocabulary size, demonstrating the impact of `vocab_size` on token granularity.

#### 7. IMDB Dataset Tokenization
*   `tokenize_function(examples)`: A function defined to process text examples from the IMDB dataset.
    *   It calls `tokenizer.encode(examples["text"])` to get the `Encoding` object.
    *   **Crucially, it returns `{"tokens": encoding.tokens, "ids": encoding.ids}`**, converting the `Encoding` object into a dictionary. This is necessary because `dataset.map` expects a dictionary to add new columns to the dataset.
*   `tokenized_imdb_train = imdb_dataset[0].map(tokenize_function, batched=False)`: Applies the `tokenize_function` to the IMDB training split.
*   `tokenized_imdb_test = imdb_dataset[1].map(tokenize_function, batched=False)`: Applies the `tokenize_function` to the IMDB test split.
*   **Calculation of Total Tokens:** Sums the lengths of the `ids` lists across all entries in both `tokenized_imdb_train` and `tokenized_imdb_test` to get the total token count for the entire IMDB dataset. This provides a quantitative measure of the tokenization output.

In [ ]:
with open('/content/summary.md', 'w') as f:
    f.write("""# Notebook Summary: Deep Learning Practice with Tokenizers

This notebook demonstrates the process of building and evaluating a Byte Pair Encoding (BPE) tokenizer using the `tokenizers` library, focusing on practical implementation with both the BookCorpus and IMDB datasets.

### Theory and Mathematical Concepts

#### 1. Dataset Preparation and Sampling
*   **BookCorpus Dataset:** A large dataset used for language modeling. For efficiency, only every 7th sample is selected to create a smaller, representative subset for tokenizer training (approximately 10.5 million samples from the original 74 million). This is a form of data subsampling to manage computational resources while retaining data diversity.
*   **IMDB Dataset:** A dataset commonly used for sentiment analysis, consisting of movie reviews. It is explicitly loaded with 'train' and 'test' splits, excluding the 'unsupervised' portion to focus on supervised learning tasks if applicable, or simply reduce data volume.

#### 2. Byte Pair Encoding (BPE) Tokenization
*   **Principle:** BPE is a subword tokenization algorithm. It works by iteratively merging the most frequent adjacent character pairs into new tokens until a desired vocabulary size is reached. This allows the tokenizer to handle out-of-vocabulary (OOV) words by breaking them down into known subword units.
*   **Normalizer (Lowercase):** Before tokenization, all text is converted to lowercase. This helps reduce vocabulary size and treats words like "The" and "the" as the same token, simplifying the model's learning process.
*   **Pre-tokenizer (Whitespace):** The text is initially split into words based on whitespace. BPE then operates on these words or character sequences. This is crucial for defining initial token boundaries.
*   **Special Tokens:** Specific tokens are added to the vocabulary to denote special semantic meanings or boundaries, such as:
    *   `[GO]` (Go/Beginning of Sentence)
    *   `[UNK]` (Unknown): For tokens not present in the vocabulary.
    *   `[PAD]` (Padding): To standardize sequence lengths.
    *   `[EOS]` (End of Sentence)
*   **Decoder (BPEDecoder):** Responsible for reconstructing the original text from token IDs. With a `Whitespace` pre-tokenizer and `BPEDecoder` (without explicit post-processing), the decoded text will appear concatenated without spaces. This is because the tokenizer handles spaces as part of the token boundaries during pre-tokenization but doesn't re-insert them during simple decoding unless specific post-processing rules are applied.
*   **Vocabulary Size:** The number of unique tokens the tokenizer can produce. Varying vocabulary sizes (e.g., 10K, 15K, 32K) directly impacts the granularity of tokenization:
    *   **Smaller vocab:** More subword tokens, shorter tokens, more common words are split (e.g., 'traders' -> 'trad', 'ers').
    *   **Larger vocab:** Fewer, longer tokens, more common words are represented as single tokens (e.g., 'traders' as a single token). This often leads to more efficient representation for frequent words.

### Code Implementation and Practice

#### 1. Environment Setup
*   Installation of `tokenizers` library using `!pip install tokenizers`.
*   Setting a Hugging Face token (`HF_TOKEN`) for dataset access.

#### 2. Dataset Loading
*   `load_dataset("rojagtap/bookcorpus")`: Loads the BookCorpus dataset.
*   `ds['train'].select(indices)`: Selects every 7th sample from the BookCorpus training split.
*   `load_dataset("stanfordnlp/imdb", split=['train', 'test'])`: Loads the IMDB dataset, specifically the train and test splits.

#### 3. Tokenizer Initialization and Configuration
*   `from tokenizers import Tokenizer`: Imports the main `Tokenizer` class.
*   `from tokenizers.models import BPE`: Imports the BPE model.
*   `from tokenizers.pre_tokenizers import Whitespace`: Imports the whitespace pre-tokenizer.
*   `from tokenizers.normalizers import Lowercase`: Imports the lowercase normalizer.
*   `from tokenizers import decoders`: Imports decoders.
*   `tokenizer = Tokenizer(BPE(unk_token=special_tokens["unk_token"]))`: Initializes the tokenizer with BPE model and an unknown token.
*   `tokenizer.normalizer = Lowercase()`: Sets the normalizer.
*   `tokenizer.pre_tokenizer = Whitespace()`: Sets the pre-tokenizer.
*   `tokenizer.decoder = decoders.BPEDecoder()`: Sets the decoder to BPEDecoder.

#### 4. Tokenizer Training
*   `batch_iterator` function: A Python generator that yields batches of text from the dataset. This is essential for training the tokenizer from a large dataset efficiently.
*   `from tokenizers.trainers import BpeTrainer`: Imports the BPE trainer class.
*   `trainer = BpeTrainer(...)`: Initializes the trainer with `vocab_size`, `min_frequency`, and `special_tokens`.
*   `tokenizer.train_from_iterator(batch_iterator(ds_copy), trainer=trainer, length=len(ds_copy))`: Trains the BPE tokenizer using the prepared BookCorpus subset.

#### 5. Tokenization and Decoding Sample Text
*   `input_text = "SEBI study finds..."`: Defines a sample text.
*   `output = tokenizer.encode(input_text)`: Tokenizes the input text, returning an `Encoding` object containing tokens and their IDs.
*   `tokenizer.decode(output.ids)`: Decodes the token IDs back to text.

#### 6. Retraining with Varying Vocabulary Sizes
*   A loop iterates through `vocab_sizes_to_test = [10000, 15000, 32000]`.
*   Inside the loop, the tokenizer and trainer are re-initialized and retrained with each specified vocabulary size.
*   The same sample text is tokenized and decoded for each vocabulary size, demonstrating the impact of `vocab_size` on token granularity.

#### 7. IMDB Dataset Tokenization
*   `tokenize_function(examples)`: A function defined to process text examples from the IMDB dataset.
    *   It calls `tokenizer.encode(examples["text"])` to get the `Encoding` object.
    *   **Crucially, it returns `{"tokens": encoding.tokens, "ids": encoding.ids}`**, converting the `Encoding` object into a dictionary. This is necessary because `dataset.map` expects a dictionary to add new columns to the dataset.
*   `tokenized_imdb_train = imdb_dataset[0].map(tokenize_function, batched=False)`: Applies the `tokenize_function` to the IMDB training split.
*   `tokenized_imdb_test = imdb_dataset[1].map(tokenize_function, batched=False)`: Applies the `tokenize_function` to the IMDB test split.
*   **Calculation of Total Tokens:** Sums the lengths of the `ids` lists across all entries in both `tokenized_imdb_train` and `tokenized_imdb_test` to get the total token count for the entire IMDB dataset. This provides a quantitative measure of the tokenization output.""")

False